# Poker Chip Stack Segmentation Training (YOLOv8 + Roboflow)

This notebook is designed to run on **Google Colab** to:
1. Download the dataset from Roboflow (COCO segmentation format)
2. Convert COCO annotations to YOLO segmentation labels
3. Train a YOLOv8 segmentation model
4. Run inference on a few validation images
5. Export the trained model to ONNX for browser use


## Setup

In Colab: go to **Runtime -> Change runtime type -> GPU** for faster training.


In [1]:
# Install required packages:
# - roboflow: dataset download
# - ultralytics: YOLOv8 training/inference/export
# - pycocotools: read COCO JSON annotations
!pip -q install roboflow ultralytics pycocotools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 58.2 MB/s eta 0:00:00


In [2]:
# Optional: verify that GPU is available in Colab.
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Download dataset from Roboflow (COCO Segmentation)

As per Roboflow instructions.


In [5]:
from os import environ
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace('yanns-workspace-ufdsd').project('poker-chip-stacks-j395l')
version = project.version(3)
dataset = version.download('yolov8')

print('Dataset downloaded to:', dataset.location)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Poker-Chip-Stacks-3 in yolov8:: 100%|██████████| 1677/1677 [00:00<00:00, 6419.45it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset downloaded to: /content/Poker-Chip-Stacks-3


## Train a YOLOv8 segmentation model

We start from pretrained `yolov8n-seg.pt` (nano) for speed.
You can switch to `yolov8s-seg.pt` (small) or bigger models for better quality.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

download_root = Path(dataset.location)
#yolo_dataset_root = download_root / 'train'
data_yaml_path = download_root / 'data.yaml'

# Choose a segmentation checkpoint (nano is fastest).
model = YOLO('yolov8n-seg.pt')

# Train with reasonable Colab defaults.
# Increase epochs/imgsz for better performance if needed.
train_results = model.train(
    data=str(data_yaml_path),
    task='segment',
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,  # GPU in Colab
    project='runs',
    name='poker_chip_seg',
    exist_ok=True,
)

print('Training complete. Best weights at:', train_results.save_dir)


Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Poker-Chip-Stacks-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=poker_chip_seg, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

## Test inference on a few validation images


In [ ]:
import random
from IPython.display import Image, display

# Load best checkpoint produced by training.
best_weights_path = Path(train_results.save_dir) / 'weights' / 'best.pt'
best_model = YOLO(str(best_weights_path))

val_dir = download_root / 'valid' / 'images'
val_images = [p for p in val_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}]

num_samples = min(5, len(val_images))
sample_images = random.sample(val_images, num_samples) if num_samples > 0 else []

print(f'Running inference on {len(sample_images)} validation images...')
pred_results = best_model.predict(
    source=[str(p) for p in sample_images],
    task='segment',
    save=True,
    conf=0.25,
    project='runs',
    name='inference_preview',
    exist_ok=True,
)

# Display generated prediction images (with masks/boxes overlaid).
if pred_results:
    pred_dir = Path(pred_results[0].save_dir)
    print('Prediction output dir:', pred_dir)
    for img_path in sorted(pred_dir.glob('*'))[:num_samples]:
        display(Image(filename=str(img_path)))
else:
    print('No prediction results were generated. Check if validation images exist.')


## Export trained model to ONNX

This generates an ONNX file you can use in browser runtimes (for example with ONNX Runtime Web).


In [ ]:
# Export the trained segmentation model to ONNX.
onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    dynamic=True,
    simplify=True,
)

print('ONNX model exported to:', onnx_path)


In [ ]:
# Optional: copy final artifacts to /content for easy Colab download from file browser.
from pathlib import Path

artifacts = [
    best_weights_path,
    Path(onnx_path),
]

for a in artifacts:
    if a.exists():
        print('Artifact:', a)

print('Done.')
